<a href="https://colab.research.google.com/github/busycaesar/Embeddings_And_Cosine_Similarity/blob/Master/TorontoJS/main.colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install required dependencies

In [15]:
%pip install pypdf langchain-community langchain-google-community langchain_huggingface langchain-google-genai requests==2.32.4

Import environment variables

In [4]:
from google.colab import userdata

GEMINI_API_KEYS = userdata.get('GEMINI_API_KEYS')
PROJECT_ID = userdata.get('PROJECT_ID')
DATASET = userdata.get('DATASET')
TABLE = userdata.get('TABLE')
REGION = userdata.get('REGION')

from google.colab import auth
auth.authenticate_user()

Fetch the data

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://services.google.com/fh/files/misc/startup_technical_guide_ai_agents_final.pdf")

documents = loader.load()

Split the data into chunks

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)

chunks = text_splitter.split_documents(documents)

Store the data into vector database

In [9]:
from langchain_google_community import BigQueryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

bq_vector_store = BigQueryVectorStore(
    project_id=PROJECT_ID,
    dataset_name=DATASET,
    table_name=TABLE,
    location=REGION,
    embedding=embedding_model
)

bq_vector_store.add_documents(chunks)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO:langchain_google_community.bq_storage_vectorstores._base:BigQuery table torontojs-488120.torontojs_vectordb.content initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=torontojs-488120&ws=!1m5!1m4!4m3!1storontojs-488120!2storontojs_vectordb!3scontent


['ff1021ac701f4a5797d2254b3789e12c',
 '02bac1c15c74487e866da2d692d2a9d3',
 '6d815c67401f41b7826e91366e365b43',
 '78dbcbef51bb4bc4a1dcb7147d144c76',
 '1989355960fb4680aa1062ce2247d41c',
 '9e64a1b663ce46f19c0852253629a541',
 'c319ca3efdb24daa81518e141939ceb0',
 'bdbbdf8ef1104fac9e487f99162c38e7',
 '82f6b9deaac64f9fa3d4c3f9e54c7baa',
 '1027463f28b64ccf8b124523b7a18a24',
 '7ef8d21c96534b7ca76a1fb6be959371',
 'fde62237b6684473a0594d3dcf4667aa',
 '004d9e1671f74b79b3d132bf2fead2a6',
 '2cc6348def0e43dc82dd2d53607adcf3',
 '5560daeaedd6491fb6126c39e6bc8d6d',
 '4d2040c4b16e4804890bead92b48853e',
 '82920105fc174716955864c47962cda6',
 '5b325fcced9e4dcbaf35659cbc0124c5',
 'cf0b7f45817b4c5e856ac20c8104590a',
 '66c6a5381003488bab17d2b9c893bcde',
 '665bc7ac58104899bf998004d3390f73',
 'e0ff79bc2b544a21927f976069657c31',
 'acb8fa13d33249f99942ba3b3de906be',
 '64435adcdb1c4b86a50b441f2d619534',
 'f2df0235eb8f46d4a192960ef2ceb4e5',
 '50f262ca5e22489fa059641d9c023654',
 '483f506058de4f5a8689b1e3bd8e5a6a',
 

User's query

In [10]:
user_query = "What are the Core components for building AI agents?"

Fetch the relevant chunk of data

In [11]:
retrieved_docs = bq_vector_store.as_retriever().invoke(user_query)

retrieved_docs = " ".join([doc.page_content for doc in retrieved_docs])

Create prompt template

In [12]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["prompt", "relevant_chunk_of_data"],
    template=
    """
        Use the following pieces of context to answer the question at the end.

        Context: {relevant_chunk_of_data}

        User's Question: {prompt}
    """
)

Chain the prompt template with LLM and invoke it to generate the response

In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI
from IPython.display import clear_output

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", google_api_key=GEMINI_API_KEYS)

chain = prompt_template | llm

response = chain.invoke({
    "prompt": user_query,
    "relevant_chunk_of_data": retrieved_docs
})

clear_output(wait=True)

print(response.content)

The core components for building AI agents are:

*   A core reasoning model
*   A set of tools to enable action
*   Data architecture options for short-term and long-term agent memory
*   A grounding mechanism for factual accuracy
*   Deployment options
